<a href="https://colab.research.google.com/github/GU-DPM/DPM_DB/blob/main/oldCode/Process_misspecification_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()

project_id = 'mccoylab'
!gcloud config set project {project_id}
sim_output_bucket = 'gs://misspecification_dpm/'
dbFile_output_dir = 'gs://misspecification_dpm_db_files/'
failed_sim_output_dir = '/content/drive/MyDrive/DPM_misspecification/failed_sim_outputs/'

Mounted at /content/drive
INFORMATION: Project 'mccoylab' has no 'environment' tag set. Use either 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`.
Updated property [core/project].


In [ ]:
import os
import pandas as pd
import math
import numpy as np
from collections import defaultdict
import pandas as pd
import pandas as pd
import re

def process_group_output(group_output_data):
    # Regex to match headers like "(x1 dosage, x2) at t=123"
    header_pattern = re.compile(r"\((.*?)\)\s+at\s+t=(\d+)")

    def expand_columns(df):
        new_df = df.copy()
        new_columns = []

        for col in df.columns:
            match = header_pattern.match(col)
            if match:
                var_names = [v.strip() for v in match.group(1).split(",")]
                t_number = match.group(2)

                expanded_values = df[col].apply(lambda x: [v.strip() for v in str(x).strip("()").split(",")])

                temp_data = {}
                for i, var in enumerate(var_names):
                    if " dosage" in var:
                        var = var.replace(" dosage", "")
                    else:
                        var = var + "pop"

                    new_col_name = f"{var}_{t_number}"
                    temp_data[new_col_name] = expanded_values.apply(lambda vals: vals[i] if i < len(vals) else None)

                new_columns.append(pd.DataFrame(temp_data))
                new_df.drop(columns=[col], inplace=True)

        if new_columns:
            new_df = pd.concat([new_df] + new_columns, axis=1)

        return new_df

    # Expand dosage and pop
    expanded_dosage = expand_columns(group_output_data['dosage'])
    expanded_dosage.rename(columns={'paramID': "Parameter_ID", 'Strategy name': "Strategy_name"}, inplace=True)

    expanded_pop = expand_columns(group_output_data['pop'])
    expanded_pop.rename(columns={'paramID': "Parameter_ID", 'Strategy name': "Strategy_name"}, inplace=True)

    # Rename para columns
    param_col_names = {
        'paramID': "Parameter_ID",
        'Spop': 'S_pop',
        'R1pop': 'R1_pop',
        'R2pop': 'R2_pop',
        'R12pop': 'R12_pop',
        'g0_S': 'g0',
        'Sa.S.D1.': 'S_cell_sensitivity_D1',
        'Sa.S.D2.': 'S_cell_sensitivity_D2',
        'Sa.R1.D1.': 'R1_cell_sensitivity_D1',
        'Sa.R1.D2.': 'R1_cell_sensitivity_D2',
        'Sa.R2.D1.': 'R2_cell_sensitivity_D1',
        'Sa.R2.D2.': 'R2_cell_sensitivity_D2',
        'Sa.R12.D1.': 'R12_cell_sensitivity_D1',
        'Sa.R12.D2.': 'R12_cell_sensitivity_D2',
        'T.S..S.': 'S_transition_to_S',
        'T.S..R1.': 'R1_transition_to_S',
        'T.S..R2.': 'R2_transition_to_S',
        'T.S..R12.': 'R21_transition_to_S',
        'T.R1..S.': 'S_transition_to_R1',
        'T.R1..R1.': 'R1_transition_to_R1',
        'T.R1..R2.': 'R2_transition_to_R1',
        'T.R1..R12.': 'R12_transition_to_R1',
        'T.R2..S.': 'S_transition_to_R2',
        'T.R2..R1.': 'R1_transition_to_R2',
        'T.R2..R2.': 'R2_transition_to_R2',
        'T.R2..R12.': 'R12_transition_to_R2',
        'T.R12..S.': 'S_transition_to_R12',
        'T.R12..R1.': 'R1_transition_to_R12',
        'T.R12..R2.': 'R2_transition_to_R12',
        'T.R12..R12.': 'R12_transition_to_R12'
    }
    renamed_param = group_output_data['para'].rename(columns=param_col_names)

    # Rename stopt columns
    stopt_column_names = ['Parameter_ID', 'Survival_CPM', 'Survival_DPM']
    renamed_stopt = group_output_data['stopt'].copy()
    renamed_stopt.columns = stopt_column_names

    return {
        "expanded_dosage": expanded_dosage,
        "expanded_pop": expanded_pop,
        "renamed_param": renamed_param,
        "renamed_stopt": renamed_stopt
    }

def get_group_ids(file_paths):
    group_ids = set()
    for path in file_paths:
        filename = path.split('/')[-1]
        fields = filename.split('_')
        if len(fields) >= 6:
            sixth_field = f"_{fields[5]}_"
            group_ids.add(sixth_field)
    return group_ids

def validate_groups(file_paths):
    required_fields = {"dosage", "para", "pop", "stopt"}
    group_map = defaultdict(set)

    for path in file_paths:
        filename = path.split('/')[-1]
        fields = filename.split('_')
        if len(fields) >= 10:
            group_id = f"_{fields[5]}_"
            tenth_field = fields[9]
            if tenth_field in required_fields:
                group_map[group_id].add(tenth_field)

    valid_groups = {gid for gid, fields in group_map.items() if fields == required_fields}
    return list(valid_groups)

def load_group_dataframes(file_paths, target_group_id):
    required_fields = {"dosage", "para", "pop", "stopt"}
    dataframes = {}

    for path in file_paths:
        filename = os.path.basename(path)
        fields = filename.split('_')
        if len(fields) >= 10:
            group_id = f"_{fields[5]}_"
            category = fields[9].split('.')[0]
            if group_id == target_group_id and category in required_fields:
                df = pd.read_csv(path)#.dropna()
                if category in {"dosage", "pop"}:
                    df = df[df["Strategy name"].isin(["strategy0", "strategy2.2"])]
                    df = df.iloc[:, :42]
                elif category == "stopt":
                    columns_to_keep = ["paramID", "strategy0", "strategy2.2"]
                    df = df[columns_to_keep]
                dataframes[category] = df
    return dataframes

def show_dataframe_columns(data_dict):
    for key, df in data_dict.items():
        if hasattr(df, 'columns'):  # Ensure the value is a DataFrame-like object
            print(f"Columns for '{key}':")
            print(list(df.columns))
            print("-" * 40)
        else:
            print(f"Value for '{key}' is not a DataFrame.")
            print("-" * 40)

def round_to_significant_digits(value, digits):
    if value == 0:
        return 0
    else:
        return round(value, digits - int(math.floor(math.log10(abs(value)))) - 1)

def gen_input_params(param_df):

  input_params_dict = {'Parameter_ID' : param_df['Parameter_ID'],
                       'initial_R1_percent' : [x/5e9 for x in param_df['R1_pop'] ],
                       'initial_R2_percent' : [x/5e9 for x in param_df['R2_pop'] ],
                       'g0' : param_df['g0'],
                       'S_sensitivity_D1_to_g0' : param_df['S_cell_sensitivity_D1']/param_df['g0'],
                       'S_sensitivity_D2_to_S_sensitivity_D1' : param_df['S_cell_sensitivity_D2']/param_df['S_cell_sensitivity_D1'],
                       'R1_sensitivity_D1_to_S_sensitivity_D1' : param_df['R1_cell_sensitivity_D1']/param_df['S_cell_sensitivity_D1'],
                       'R2_sensitivity_D2_to_S_sensitivity_D2' : param_df['R2_cell_sensitivity_D2']/param_df['S_cell_sensitivity_D2'],
                       'S_transition_to_R1': param_df['S_transition_to_R1'],
                       'S_transition_to_R2': param_df['S_transition_to_R2'],
                       }
  return pd.DataFrame(input_params_dict)

import math
def round_to_significant_digits(value, digits):
    if value == 0:
        return 0
    else:
        return round(value, digits - int(math.floor(math.log10(abs(value)))) - 1)

def check_input_params(input_param_df):
  # check if params are valid inputs
  valid_input_values = {'initial_R1_percent' : [0, 1e-09, 1e-07, 1e-05, 0.001, 0.1, 0.9],
                        'initial_R2_percent' : [0, 1e-09, 1e-07, 1e-05, 0.001, 0.1, 0.9],
                        'g0' : [0.001, 0.0026, 0.007, 0.0184, 0.0487, 0.129, 0.34],
                        'S_sensitivity_D1_to_g0' : [5.6e-4, 0.0054, 0.0517, 0.496, 4.77, 45.8, 440.0],
                        'S_sensitivity_D2_to_S_sensitivity_D1' : [0.0004, 0.0015, 0.0054, 0.02, 0.0737, 0.271, 1.0],
                        'R1_sensitivity_D1_to_S_sensitivity_D1' : [0, 1e-05, 9.56e-05, 0.000915, 0.0087, 0.0837, 0.8],
                        'R2_sensitivity_D2_to_S_sensitivity_D2' : [0, 1e-05, 9.56e-05, 0.000915, 0.0087, 0.0837, 0.8],
                        'S_transition_to_R1' : [1e-11, 2.15e-10, 4.64e-09, 1e-07, 2.15e-06, 4.64e-05, 0.001],
                        'S_transition_to_R2' : [1e-11, 2.15e-10, 4.64e-09, 1e-07, 2.15e-06, 4.64e-05, 0.001],
                        }

  valid_parameters = {'Parameter_ID' : input_param_df['Parameter_ID'] }
  for input_param in valid_input_values.keys():
    if input_param in input_param_df.columns:
      input_param_df[input_param] = input_param_df[input_param].apply(lambda x: round_to_significant_digits(x,3))
      valid_parameters[input_param] = input_param_df[input_param].apply(lambda x: x in valid_input_values[input_param])
  invalid_parameters_df = pd.DataFrame(valid_parameters)
  invalid_parameters_df = invalid_parameters_df[invalid_parameters_df.applymap(lambda x: x is False).any(axis=1)]
  return invalid_parameters_df



def merge_input_output_params(output_param_df, input_param_df):
  return pd.merge(input_param_df, output_param_df, on='Parameter_ID', how='inner')

def map_parameters(sim_run_id, misspec_rate, param_df, mapped_dir):#(sim_run_id, results_dir, mapped_dir):
  print(str(sim_run_id))
  print('Generating input parameter from output set')
  input_param_df = gen_input_params(param_df)
  #print(input_param_df.head())
  if not os.path.exists(os.path.join(mapped_dir,misspec_rate + "_" + str(sim_run_id) + '_simParamOutput.csv')):
    print("Generateing Output Param CSV")
    param_df["misspecification_rate"] = misspec_rate
    param_df.to_csv(os.path.join(mapped_dir,misspec_rate + "_" + str(sim_run_id) + '_simParamOutput.csv'), header=True, index=False)
  if not os.path.exists(os.path.join(mapped_dir,misspec_rate + "_" + str(sim_run_id) + '_inputParam.csv')):
    input_param_df["misspecification_rate"] = misspec_rate
    input_param_df.to_csv(os.path.join(mapped_dir,misspec_rate + "_" + str(sim_run_id) + '_inputParam.csv'), header=True, index=False)
  if not os.path.exists(os.path.join(mapped_dir,misspec_rate + "_" + str(sim_run_id) + '_invalidParams.csv')):
    invalid_params_df = check_input_params(input_param_df)
    if len(invalid_params_df) > 0:
      invalid_params_df["misspecification_rate"] = misspec_rate
      invalid_params_df.to_csv(os.path.join(mapped_dir,misspec_rate + "_" + str(sim_run_id) + '_invalidParams.csv'), header=True, index=False)

def get_dosage_df(dosage_df):
  # only need drug 1 since it defines drug 2 dosage
  melted_dosage_df = dosage_df.melt(id_vars=['Parameter_ID', 'Strategy_name'], value_vars=[col for col in dosage_df.columns if 'Drug1' in col], var_name='timepoint', value_name='Drug1_dosage')
  melted_dosage_df['timepoint'] = melted_dosage_df['timepoint'].str.extract(r'_(\d+)').astype(int)

  # add t = 1800
  new_rows = melted_dosage_df[['Parameter_ID', 'Strategy_name']].drop_duplicates()
  new_rows['timepoint'] = 1800
  new_rows['Drug1_dosage'] = -1

  melted_dosage_df = pd.concat([melted_dosage_df, new_rows], ignore_index=True)
  melted_dosage_df = melted_dosage_df.sort_values(by=['Parameter_ID', 'Strategy_name', 'timepoint']).reset_index(drop=True)

  strategy_map = { 'strategy0' : 'CPM', 'strategy2.2' : 'DPM'}
  melted_dosage_df['Strategy_name'] = melted_dosage_df['Strategy_name'].replace(strategy_map)
  #print(melted_dosage_df.head())

  return melted_dosage_df


def get_pop_df(pop_df, initial_pop_df):
  strategy_map = { 'strategy0' : 'CPM', 'strategy2.2' : 'DPM'}
  pop_df['Strategy_name'] = pop_df['Strategy_name'].replace(strategy_map)

  # add t = 0
  initial_pop_df = initial_pop_df.rename(columns = {'S_pop':'Spop_0',
                                                    'R1_pop':'R1pop_0',
                                                    'R2_pop':'R2pop_0',
                                                    'R12_pop':'R12pop_0',})

  pop_df = pd.merge(pop_df, initial_pop_df, on = (['Parameter_ID']))

  melted_Spop_df = pop_df.melt(id_vars=['Parameter_ID', 'Strategy_name'], value_vars=[col for col in pop_df.columns if 'Spop' in col], var_name='timepoint', value_name='Spop')
  melted_Spop_df['timepoint'] = melted_Spop_df['timepoint'].str.extract(r'_(\d+)').astype(int)
  melted_R1pop_df = pop_df.melt(id_vars=['Parameter_ID', 'Strategy_name'], value_vars=[col for col in pop_df.columns if 'R1pop' in col], var_name='timepoint', value_name='R1pop')
  melted_R1pop_df['timepoint'] = melted_R1pop_df['timepoint'].str.extract(r'_(\d+)').astype(int)
  melted_R2pop_df = pop_df.melt(id_vars=['Parameter_ID', 'Strategy_name'], value_vars=[col for col in pop_df.columns if 'R2pop' in col], var_name='timepoint', value_name='R2pop')
  melted_R2pop_df['timepoint'] = melted_R2pop_df['timepoint'].str.extract(r'_(\d+)').astype(int)
  melted_R12pop_df = pop_df.melt(id_vars=['Parameter_ID', 'Strategy_name'], value_vars=[col for col in pop_df.columns if 'R12pop' in col], var_name='timepoint', value_name='R12pop')
  melted_R12pop_df['timepoint'] = melted_R12pop_df['timepoint'].str.extract(r'_(\d+)').astype(int)

  merged_pop_df = pd.merge(melted_Spop_df,melted_R1pop_df, on=['Parameter_ID', 'Strategy_name', 'timepoint'])
  merged_pop_df = pd.merge(merged_pop_df,melted_R2pop_df, on=['Parameter_ID', 'Strategy_name', 'timepoint'])
  merged_pop_df = pd.merge(merged_pop_df,melted_R12pop_df, on=['Parameter_ID', 'Strategy_name', 'timepoint'])


  #melted_dosage_df = pd.concat([melted_dosage_df, new_rows], ignore_index=True)
  merged_pop_df = merged_pop_df.sort_values(by=['Parameter_ID', 'Strategy_name', 'timepoint']).reset_index(drop=True)
  #print(merged_pop_df.head())

  return merged_pop_df


def map_trajectories(sim_run_id, misspec_rate, param_df, dosage_df, pop_df, output_dir, overwrite_file = False):
  pop_df = get_pop_df(pop_df, param_df[['Parameter_ID','S_pop','R1_pop','R2_pop','R12_pop']])
  dosage_df = get_dosage_df(dosage_df)
  #print(dosage_df.head())
  #print(pop_df.head())
  traj_df = pd.merge(dosage_df, pop_df, on =['Parameter_ID', 'Strategy_name', 'timepoint'] )
  #print(traj_df.head())
  traj_df['misspecification_rate'] = misspec_rate
  traj_df.to_csv(os.path.join(output_dir,misspec_rate + "_" + str(sim_run_id) + '_simTrajectories.csv'), header = True, index = False)

def collect_EC_and_survival(sim_run_id, misspec_rate, stopt_df, output_dir):
  if not os.path.exists(os.path.join(output_dir,misspec_rate + "_" + str(sim_run_id) + '_ECsurvival.csv')):
    if os.path.exists(os.path.join(output_dir,misspec_rate + "_" + str(sim_run_id) + '_simTrajectories.csv')):
      run_trajectory_df = pd.read_csv(os.path.join(output_dir,misspec_rate + "_" + str(sim_run_id) + '_simTrajectories.csv'))
    else:
        print('no _simTrajectories.csv... run map_trajectories() first')
    param_id_list = []
    category_list = []
    basket_list = []
    for param_id in run_trajectory_df['Parameter_ID'].unique():
      param_id_list.append(param_id)
      param_id_check = run_trajectory_df['Parameter_ID'] == param_id
      time_0_check = run_trajectory_df['timepoint'] == 0
      time_45_check = run_trajectory_df['timepoint'] == 45

      param_trajectory_0_df = run_trajectory_df[param_id_check & time_0_check]
      param_trajectory_45_df = run_trajectory_df[param_id_check & time_45_check]
      #print(param_trajectory_df)
      CPM_drug_0 = param_trajectory_0_df[param_trajectory_0_df['Strategy_name'] == 'CPM']['Drug1_dosage'].iloc[0]
      DPM_drug_0 = param_trajectory_0_df[param_trajectory_0_df['Strategy_name'] == 'DPM']['Drug1_dosage'].iloc[0]
      CPM_drug_45 = param_trajectory_45_df[param_trajectory_45_df['Strategy_name'] == 'CPM']['Drug1_dosage'].iloc[0]
      DPM_drug_45 = param_trajectory_45_df[param_trajectory_45_df['Strategy_name'] == 'DPM']['Drug1_dosage'].iloc[0]

      if CPM_drug_0 == DPM_drug_0 and CPM_drug_45 == DPM_drug_45:
        category_list.append('both_same')
      elif CPM_drug_0 == DPM_drug_0 and CPM_drug_45 != DPM_drug_45:
        category_list.append('first_same_only')
      elif CPM_drug_0 != DPM_drug_0 and CPM_drug_45 == DPM_drug_45:
        category_list.append('second_same_only')
      else:
        category_list.append('both diff')

      basket_list.append(str(CPM_drug_0) + '_' + str(DPM_drug_0) + '_' + str(CPM_drug_45) + '_' + str(DPM_drug_45))

    category_df = pd.DataFrame({'Parameter_ID' : param_id_list,
                                'EC_category' : category_list,
                                'Bucket' : basket_list})

    survival_df = pd.merge(stopt_df,category_df, on = ['Parameter_ID'])
    survival_df['DPM_days_improvement'] = survival_df['Survival_DPM'] - survival_df['Survival_CPM']
    survival_df['DPM_percent_improvement'] = (survival_df['Survival_DPM'] - survival_df['Survival_CPM'])/survival_df['Survival_DPM']

    survival_df['misspecification_rate'] = misspec_rate
    survival_df.to_csv(os.path.join(output_dir,misspec_rate + "_" + str(sim_run_id) + '_ECsurvival.csv'), header = True, index = False)

def process_sim_output(sim_run_id, misspec_rate, param_df, dosage_df, pop_df, stopt_df, output_dir, overwrite_file = False):
  map_parameters(sim_run_id, misspec_rate, param_df, output_dir)
  map_trajectories(sim_run_id, misspec_rate,  param_df, dosage_df, pop_df, output_dir, overwrite_file)
  collect_EC_and_survival(sim_run_id, misspec_rate, stopt_df, output_dir)

# testing and dev

In [ ]:
# get miss rate from simulation output folders
sim_output_dirs = !gsutil ls {sim_output_bucket}
misspecification_rates = []

for path in sim_output_dirs:
    final_part = path.rstrip('/').split('/')[-1]
    miss_rate = final_part.split('_')[0]
    misspecification_rates.append(miss_rate)

In [ ]:
# select misspecificaiton rate simulation
test_miss_rate = misspecification_rates[0]

In [ ]:
# get sim output groups
test_sim_output_files = !gsutil ls gs://misspecification_dpm/{test_miss_rate}_atsim/
valid_groups = validate_groups(test_sim_output_files)
print(len(valid_groups)," Valid groups with all required fields:", valid_groups)

290  Valid groups with all required fields: ['_005_', '_054_', '_087_', '_091_', '_111_', '_233_', '_044_', '_298_', '_248_', '_295_', '_289_', '_097_', '_038_', '_070_', '_162_', '_145_', '_085_', '_025_', '_282_', '_149_', '_246_', '_231_', '_254_', '_100_', '_300_', '_131_', '_115_', '_240_', '_041_', '_006_', '_079_', '_113_', '_104_', '_153_', '_287_', '_166_', '_213_', '_245_', '_288_', '_210_', '_280_', '_049_', '_190_', '_201_', '_292_', '_067_', '_171_', '_147_', '_215_', '_234_', '_253_', '_109_', '_258_', '_089_', '_057_', '_008_', '_069_', '_018_', '_062_', '_095_', '_036_', '_077_', '_226_', '_096_', '_150_', '_238_', '_256_', '_304_', '_032_', '_185_', '_116_', '_031_', '_013_', '_066_', '_291_', '_262_', '_290_', '_014_', '_080_', '_022_', '_272_', '_010_', '_142_', '_216_', '_112_', '_148_', '_034_', '_072_', '_092_', '_159_', '_167_', '_028_', '_135_', '_227_', '_275_', '_136_', '_099_', '_236_', '_267_', '_297_', '_122_', '_108_', '_045_', '_120_', '_114_', '_035_', '

In [ ]:
# select output group
test_output_group = valid_groups[0]
group_paths = !gsutil ls {sim_output_bucket}{test_miss_rate}_atsim/*{test_output_group}*
group_paths

['gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_dosage_20250107.csv',
 'gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_para_20250107.csv',
 'gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_pop_20250110.csv',
 'gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_stopt_20250107.csv']

## raw simulation outputs

In [ ]:

for url in group_paths:
    !gsutil cp {url} .


Copying gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_dosage_20250107.csv...
/ [1 files][  6.8 MiB/  6.8 MiB]                                                
Operation completed over 1 objects/6.8 MiB.                                      
Copying gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_para_20250107.csv...
/ [1 files][  1.7 MiB/  1.7 MiB]                                                
Operation completed over 1 objects/1.7 MiB.                                      
Copying gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_pop_20250110.csv...
/ [1 files][ 12.8 MiB/ 12.8 MiB]                                                
Operation completed over 1 objects/12.8 MiB.                                     
Copying gs://misspecification_dpm/10x_atsim/mis_PARset_default_PNAS_2drug_005_10000_para_result_stopt_20250107.csv...
/ [1 files][551.3 KiB/551.3 KiB]         

In [ ]:
raw_dosage_df = pd.read_csv(os.path.basename(group_paths[0]))
raw_para_df = pd.read_csv(os.path.basename(group_paths[1]))
raw_pop_df = pd.read_csv(os.path.basename(group_paths[2]))
raw_stopt_df = pd.read_csv(os.path.basename(group_paths[3]))

In [ ]:
raw_dosage_df.head()

,paramID,Strategy name,"(Drug1 dosage,Drug2 dosage) at t=0","(Drug1 dosage,Drug2 dosage) at t=45","(Drug1 dosage,Drug2 dosage) at t=90","(Drug1 dosage,Drug2 dosage) at t=135","(Drug1 dosage,Drug2 dosage) at t=180","(Drug1 dosage,Drug2 dosage) at t=225","(Drug1 dosage,Drug2 dosage) at t=270","(Drug1 dosage,Drug2 dosage) at t=315",...,"(Drug1 dosage,Drug2 dosage) at t=1350","(Drug1 dosage,Drug2 dosage) at t=1395","(Drug1 dosage,Drug2 dosage) at t=1440","(Drug1 dosage,Drug2 dosage) at t=1485","(Drug1 dosage,Drug2 dosage) at t=1530","(Drug1 dosage,Drug2 dosage) at t=1575","(Drug1 dosage,Drug2 dosage) at t=1620","(Drug1 dosage,Drug2 dosage) at t=1665","(Drug1 dosage,Drug2 dosage) at t=1710","(Drug1 dosage,Drug2 dosage) at t=1755"
0,1253756,strategy0,"(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
1,1253756,strategy1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1253756,strategy2.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1253756,strategy2.2,"(1.0,0.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
4,1253756,strategy3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
raw_para_df.head()
raw_para_df.columns

Index(['paramID', 'Spop', 'R1pop', 'R2pop', 'R12pop', 'g0_S', 'g0_R1', 'g0_R2',
       'g0_R12', 'Sa.S.D1.', 'Sa.S.D2.', 'Sa.R1.D1.', 'Sa.R1.D2.', 'Sa.R2.D1.',
       'Sa.R2.D2.', 'Sa.R12.D1.', 'Sa.R12.D2.', 'T.S..S.', 'T.S..R1.',
       'T.S..R2.', 'T.S..R12.', 'T.R1..S.', 'T.R1..R1.', 'T.R1..R2.',
       'T.R1..R12.', 'T.R2..S.', 'T.R2..R1.', 'T.R2..R2.', 'T.R2..R12.',
       'T.R12..S.', 'T.R12..R1.', 'T.R12..R2.', 'T.R12..R12.'],
      dtype='object')

In [ ]:
raw_pop_df.head()
raw_pop_df.columns

In [ ]:
raw_stopt_df.head()

,paramID,strategy0,strategy1,strategy2.1,strategy2.2,strategy3,strategy4,strategy5,strategy6,strategy7,strategy8,strategy9,optimal
0,1253756,1845.0,NaN,NaN,1845.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1253699,1442.0,NaN,NaN,1568.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1255036,1845.0,NaN,NaN,1845.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1255770,1306.0,NaN,NaN,1418.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1258270,1143.0,NaN,NaN,1393.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## loading relavant simulation output data

In [ ]:
group_output_data = load_group_dataframes(group_paths, test_output_group)

In [ ]:
group_output_data.keys()

dict_keys(['dosage', 'para', 'pop', 'stopt'])

In [ ]:
loaded_dosage_df = group_output_data['dosage']
loaded_para_df = group_output_data['para']
loaded_pop_df = group_output_data['pop']
loaded_stopt_df = group_output_data['stopt']

In [ ]:
loaded_dosage_df.head()

,paramID,Strategy name,"(Drug1 dosage,Drug2 dosage) at t=0","(Drug1 dosage,Drug2 dosage) at t=45","(Drug1 dosage,Drug2 dosage) at t=90","(Drug1 dosage,Drug2 dosage) at t=135","(Drug1 dosage,Drug2 dosage) at t=180","(Drug1 dosage,Drug2 dosage) at t=225","(Drug1 dosage,Drug2 dosage) at t=270","(Drug1 dosage,Drug2 dosage) at t=315",...,"(Drug1 dosage,Drug2 dosage) at t=1350","(Drug1 dosage,Drug2 dosage) at t=1395","(Drug1 dosage,Drug2 dosage) at t=1440","(Drug1 dosage,Drug2 dosage) at t=1485","(Drug1 dosage,Drug2 dosage) at t=1530","(Drug1 dosage,Drug2 dosage) at t=1575","(Drug1 dosage,Drug2 dosage) at t=1620","(Drug1 dosage,Drug2 dosage) at t=1665","(Drug1 dosage,Drug2 dosage) at t=1710","(Drug1 dosage,Drug2 dosage) at t=1755"
0,1253756,strategy0,"(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
3,1253756,strategy2.2,"(1.0,0.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)","(0.0,1.0)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
11,1253699,strategy0,"(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)",...,"(0.0,1.0)","(0.0,1.0)","(0.0,1.0)",-1,-1,-1,-1,-1,-1,-1
14,1253699,strategy2.2,"(1.0,0.0)","(0.5,0.5)","(0.0,1.0)","(1.0,0.0)","(0.0,1.0)","(1.0,0.0)","(0.0,1.0)","(1.0,0.0)",...,"(0.0,1.0)","(0.0,1.0)","(1.0,0.0)","(1.0,0.0)","(0.0,1.0)",-1,-1,-1,-1,-1
22,1255036,strategy0,"(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)","(1.0,0.0)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [ ]:
loaded_para_df.head()

,paramID,Spop,R1pop,R2pop,R12pop,g0_S,g0_R1,g0_R2,g0_R12,Sa.S.D1.,...,T.R1..R2.,T.R1..R12.,T.R2..S.,T.R2..R1.,T.R2..R2.,T.R2..R12.,T.R12..S.,T.R12..R1.,T.R12..R2.,T.R12..R12.
0,1253756,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-11,0,0,0,0,1.000000e-11,0.001000,0
1,1253699,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-03,0,0,0,0,1.000000e-03,0.000002,0
2,1255036,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-03,0,0,0,0,1.000000e-03,0.001000,0
3,1255770,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,4.642000e-05,0,0,0,0,4.642000e-05,0.001000,0
4,1258270,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-03,0,0,0,0,1.000000e-03,0.001000,0


In [ ]:
loaded_pop_df.head()

,paramID,Strategy name,"(S,R1,R2,R12) at t=45","(S,R1,R2,R12) at t=90","(S,R1,R2,R12) at t=135","(S,R1,R2,R12) at t=180","(S,R1,R2,R12) at t=225","(S,R1,R2,R12) at t=270","(S,R1,R2,R12) at t=315","(S,R1,R2,R12) at t=360",...,"(S,R1,R2,R12) at t=1395","(S,R1,R2,R12) at t=1440","(S,R1,R2,R12) at t=1485","(S,R1,R2,R12) at t=1530","(S,R1,R2,R12) at t=1575","(S,R1,R2,R12) at t=1620","(S,R1,R2,R12) at t=1665","(S,R1,R2,R12) at t=1710","(S,R1,R2,R12) at t=1755","(S,R1,R2,R12) at t=1800"
0,1253756,strategy0,"(2.21e+08,2.35e+06,9.50e-01,0.00e+00)","(9.60e+06,5.49e+06,9.50e-01,0.00e+00)","(4.18e+05,1.26e+07,9.50e-01,0.00e+00)","(1.82e+04,2.88e+07,9.50e-01,0.00e+00)","(7.90e+02,6.58e+07,9.50e-01,0.00e+00)","(3.44e+01,1.51e+08,9.50e-01,0.00e+00)","(1.47e+00,3.45e+08,9.50e-01,0.00e+00)","(9.80e-01,7.89e+08,9.50e-01,1.00e-02)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
3,1253756,strategy2.2,"(2.21e+08,2.35e+06,9.50e-01,0.00e+00)","(2.76e+07,3.17e+05,9.50e-01,0.00e+00)","(3.44e+06,4.24e+04,9.50e-01,0.00e+00)","(4.30e+05,5.65e+03,9.50e-01,0.00e+00)","(5.37e+04,7.50e+02,9.50e-01,0.00e+00)","(6.71e+03,9.92e+01,9.50e-01,0.00e+00)","(8.38e+02,1.31e+01,9.50e-01,0.00e+00)","(1.05e+02,1.72e+00,9.50e-01,0.00e+00)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
11,1253699,strategy0,"(2.21e+08,5.07e+03,1.83e+05,3.87e+00)","(9.78e+06,1.18e+04,1.61e+04,1.88e+01)","(4.25e+05,2.71e+04,1.07e+03,6.55e+01)","(1.85e+04,6.20e+04,6.28e+01,2.01e+02)","(8.19e+02,1.42e+05,3.46e+00,5.78e+02)","(3.62e+01,3.25e+05,9.90e-01,1.59e+03)","(1.55e+00,7.43e+05,1.00e+00,4.26e+03)","(9.80e-01,1.70e+06,1.00e+00,1.12e+04)",...,"(9.80e-01,9.70e-01,1.00e+00,4.27e+12)","(9.80e-01,9.70e-01,1.00e+00,9.77e+12)",-1,-1,-1,-1,-1,-1,-1,-1
14,1253699,strategy2.2,"(2.21e+08,5.07e+03,1.83e+05,3.87e+00)","(1.64e+07,2.80e+03,8.87e+04,1.44e+01)","(2.05e+06,3.53e+02,2.13e+05,3.51e+01)","(9.08e+04,8.11e+02,9.51e+03,8.12e+01)","(1.13e+04,1.01e+02,2.18e+04,1.86e+02)","(5.02e+02,2.32e+02,9.66e+02,4.27e+02)","(6.27e+01,2.89e+01,2.21e+03,9.77e+02)","(2.78e+00,6.62e+01,9.79e+01,2.24e+03)",...,"(9.70e-01,6.74e+08,9.70e-01,4.17e+11)","(9.70e-01,8.42e+07,9.70e-01,9.54e+11)","(9.70e-01,1.93e+08,9.70e-01,2.18e+12)","(9.70e-01,4.41e+08,9.70e-01,5.00e+12)",-1,-1,-1,-1,-1,-1
22,1255036,strategy0,"(2.21e+08,2.29e+06,1.83e+05,1.90e+03)","(9.79e+06,5.17e+06,1.61e+04,8.56e+03)","(4.26e+05,1.14e+07,1.07e+03,2.84e+04)","(1.85e+04,2.53e+07,6.29e+01,8.38e+04)","(8.19e+02,5.59e+07,3.46e+00,2.32e+05)","(3.62e+01,1.24e+08,9.90e-01,6.15e+05)","(1.55e+00,2.74e+08,1.00e+00,1.59e+06)","(9.80e-01,6.05e+08,1.00e+00,4.01e+06)",...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [ ]:
loaded_stopt_df.head()

,paramID,strategy0,strategy2.2
0,1253756,1845.0,1845.0
1,1253699,1442.0,1568.0
2,1255036,1845.0,1845.0
3,1255770,1306.0,1418.0
4,1258270,1143.0,1393.0


## processing simulation output

In [ ]:
processed_output = process_group_output(group_output_data)

In [ ]:
renamed_param = processed_output['renamed_param']
print(renamed_param.columns)
renamed_param.head()

Index(['Parameter_ID', 'S_pop', 'R1_pop', 'R2_pop', 'R12_pop', 'g0', 'g0_R1',
       'g0_R2', 'g0_R12', 'S_cell_sensitivity_D1', 'S_cell_sensitivity_D2',
       'R1_cell_sensitivity_D1', 'R1_cell_sensitivity_D2',
       'R2_cell_sensitivity_D1', 'R2_cell_sensitivity_D2',
       'R12_cell_sensitivity_D1', 'R12_cell_sensitivity_D2',
       'S_transition_to_S', 'R1_transition_to_S', 'R2_transition_to_S',
       'R21_transition_to_S', 'S_transition_to_R1', 'R1_transition_to_R1',
       'R2_transition_to_R1', 'R12_transition_to_R1', 'S_transition_to_R2',
       'R1_transition_to_R2', 'R2_transition_to_R2', 'R12_transition_to_R2',
       'S_transition_to_R12', 'R1_transition_to_R12', 'R2_transition_to_R12',
       'R12_transition_to_R12'],
      dtype='object')


,Parameter_ID,S_pop,R1_pop,R2_pop,R12_pop,g0,g0_R1,g0_R2,g0_R12,S_cell_sensitivity_D1,...,R2_transition_to_R1,R12_transition_to_R1,S_transition_to_R2,R1_transition_to_R2,R2_transition_to_R2,R12_transition_to_R2,S_transition_to_R12,R1_transition_to_R12,R2_transition_to_R12,R12_transition_to_R12
0,1253756,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-11,0,0,0,0,1.000000e-11,0.001000,0
1,1253699,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-03,0,0,0,0,1.000000e-03,0.000002,0
2,1255036,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-03,0,0,0,0,1.000000e-03,0.001000,0
3,1255770,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,4.642000e-05,0,0,0,0,4.642000e-05,0.001000,0
4,1258270,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,0,1.000000e-03,0,0,0,0,1.000000e-03,0.001000,0


In [ ]:
renamed_stopt = processed_output['renamed_stopt']
print(renamed_stopt.columns)
renamed_stopt.head()

Index(['Parameter_ID', 'Survival_CPM', 'Survival_DPM'], dtype='object')


,Parameter_ID,Survival_CPM,Survival_DPM
0,1253756,1845.0,1845.0
1,1253699,1442.0,1568.0
2,1255036,1845.0,1845.0
3,1255770,1306.0,1418.0
4,1258270,1143.0,1393.0


In [ ]:
expanded_dosage = processed_output['expanded_dosage']
print(expanded_dosage.columns)
expanded_dosage.head()

Index(['Parameter_ID', 'Strategy_name', 'Drug1_0', 'Drug2_0', 'Drug1_45',
       'Drug2_45', 'Drug1_90', 'Drug2_90', 'Drug1_135', 'Drug2_135',
       'Drug1_180', 'Drug2_180', 'Drug1_225', 'Drug2_225', 'Drug1_270',
       'Drug2_270', 'Drug1_315', 'Drug2_315', 'Drug1_360', 'Drug2_360',
       'Drug1_405', 'Drug2_405', 'Drug1_450', 'Drug2_450', 'Drug1_495',
       'Drug2_495', 'Drug1_540', 'Drug2_540', 'Drug1_585', 'Drug2_585',
       'Drug1_630', 'Drug2_630', 'Drug1_675', 'Drug2_675', 'Drug1_720',
       'Drug2_720', 'Drug1_765', 'Drug2_765', 'Drug1_810', 'Drug2_810',
       'Drug1_855', 'Drug2_855', 'Drug1_900', 'Drug2_900', 'Drug1_945',
       'Drug2_945', 'Drug1_990', 'Drug2_990', 'Drug1_1035', 'Drug2_1035',
       'Drug1_1080', 'Drug2_1080', 'Drug1_1125', 'Drug2_1125', 'Drug1_1170',
       'Drug2_1170', 'Drug1_1215', 'Drug2_1215', 'Drug1_1260', 'Drug2_1260',
       'Drug1_1305', 'Drug2_1305', 'Drug1_1350', 'Drug2_1350', 'Drug1_1395',
       'Drug2_1395', 'Drug1_1440', 'Drug2_1440',

,Parameter_ID,Strategy_name,Drug1_0,Drug2_0,Drug1_45,Drug2_45,Drug1_90,Drug2_90,Drug1_135,Drug2_135,...,Drug1_1575,Drug2_1575,Drug1_1620,Drug2_1620,Drug1_1665,Drug2_1665,Drug1_1710,Drug2_1710,Drug1_1755,Drug2_1755
0,1253756,strategy0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-1,None,-1,None,-1,None,-1,None,-1,None
3,1253756,strategy2.2,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,-1,None,-1,None,-1,None,-1,None,-1,None
11,1253699,strategy0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-1,None,-1,None,-1,None,-1,None,-1,None
14,1253699,strategy2.2,1.0,0.0,0.5,0.5,0.0,1.0,1.0,0.0,...,-1,None,-1,None,-1,None,-1,None,-1,None
22,1255036,strategy0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-1,None,-1,None,-1,None,-1,None,-1,None


In [ ]:
expanded_pop = processed_output['expanded_pop']
print(expanded_pop.columns)
expanded_pop.head()

Index(['Parameter_ID', 'Strategy_name', 'Spop_45', 'R1pop_45', 'R2pop_45',
       'R12pop_45', 'Spop_90', 'R1pop_90', 'R2pop_90', 'R12pop_90',
       ...
       'R2pop_1710', 'R12pop_1710', 'Spop_1755', 'R1pop_1755', 'R2pop_1755',
       'R12pop_1755', 'Spop_1800', 'R1pop_1800', 'R2pop_1800', 'R12pop_1800'],
      dtype='object', length=162)


,Parameter_ID,Strategy_name,Spop_45,R1pop_45,R2pop_45,R12pop_45,Spop_90,R1pop_90,R2pop_90,R12pop_90,...,R2pop_1710,R12pop_1710,Spop_1755,R1pop_1755,R2pop_1755,R12pop_1755,Spop_1800,R1pop_1800,R2pop_1800,R12pop_1800
0,1253756,strategy0,2.21e+08,2.35e+06,9.50e-01,0.00e+00,9.60e+06,5.49e+06,9.50e-01,0.00e+00,...,None,None,-1,None,None,None,-1,None,None,None
3,1253756,strategy2.2,2.21e+08,2.35e+06,9.50e-01,0.00e+00,2.76e+07,3.17e+05,9.50e-01,0.00e+00,...,None,None,-1,None,None,None,-1,None,None,None
11,1253699,strategy0,2.21e+08,5.07e+03,1.83e+05,3.87e+00,9.78e+06,1.18e+04,1.61e+04,1.88e+01,...,None,None,-1,None,None,None,-1,None,None,None
14,1253699,strategy2.2,2.21e+08,5.07e+03,1.83e+05,3.87e+00,1.64e+07,2.80e+03,8.87e+04,1.44e+01,...,None,None,-1,None,None,None,-1,None,None,None
22,1255036,strategy0,2.21e+08,2.29e+06,1.83e+05,1.90e+03,9.79e+06,5.17e+06,1.61e+04,8.56e+03,...,None,None,-1,None,None,None,-1,None,None,None


## processing processed sim output into database files

In [ ]:
sim_run_id = test_output_group.strip("_")
process_sim_output(sim_run_id, miss_rate, renamed_param, expanded_dosage, expanded_pop, renamed_stopt, "./")


005
Generating input parameter from output set
Generateing Output Param CSV


/tmp/ipython-input-1096143886.py:202: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  invalid_parameters_df = invalid_parameters_df[invalid_parameters_df.applymap(lambda x: x is False).any(axis=1)]


In [ ]:
db_inputParam_df = pd.read_csv(miss_rate + "_" + str(sim_run_id) + '_inputParam.csv')
db_inputParam_df.head()

,Parameter_ID,initial_R1_percent,initial_R2_percent,g0,S_sensitivity_D1_to_g0,S_sensitivity_D2_to_S_sensitivity_D1,R1_sensitivity_D1_to_S_sensitivity_D1,R2_sensitivity_D2_to_S_sensitivity_D2,S_transition_to_R1,S_transition_to_R2,misspecification_rate
0,1253756,0.0,1.000000e-09,0.0184,4.768315,0.736998,0.00001,0.000010,0.001000,1.000000e-11,div5
1,1253699,0.0,1.000000e-09,0.0184,4.768315,0.736998,0.00001,0.000000,0.000002,1.000000e-03,div5
2,1255036,0.0,1.000000e-09,0.0184,4.768315,0.736998,0.00870,0.800006,0.001000,1.000000e-03,div5
3,1255770,0.0,1.000000e-09,0.0184,4.768315,2.714020,0.00000,0.000000,0.001000,4.642000e-05,div5
4,1258270,0.0,1.000000e-09,0.0184,4.768315,10.000000,0.00000,0.000096,0.001000,1.000000e-03,div5


In [ ]:
db_invalidParams_df = pd.read_csv(miss_rate + "_" + str(sim_run_id) + '_invalidParams.csv')
db_invalidParams_df.head()

,Parameter_ID,initial_R1_percent,initial_R2_percent,g0,S_sensitivity_D1_to_g0,S_sensitivity_D2_to_S_sensitivity_D1,R1_sensitivity_D1_to_S_sensitivity_D1,R2_sensitivity_D2_to_S_sensitivity_D2,S_transition_to_R1,S_transition_to_R2,misspecification_rate
0,1253756,True,True,True,True,False,True,True,True,True,div5
1,1253699,True,True,True,True,False,True,True,True,True,div5
2,1255036,True,True,True,True,False,True,True,True,True,div5
3,1255770,True,True,True,True,False,True,True,True,True,div5
4,1258270,True,True,True,True,False,True,True,True,True,div5


In [ ]:
db_simParamOutput_df = pd.read_csv(miss_rate + "_" + str(sim_run_id) + '_simParamOutput.csv')
db_simParamOutput_df.head()

,Parameter_ID,S_pop,R1_pop,R2_pop,R12_pop,g0,g0_R1,g0_R2,g0_R12,S_cell_sensitivity_D1,...,R12_transition_to_R1,S_transition_to_R2,R1_transition_to_R2,R2_transition_to_R2,R12_transition_to_R2,S_transition_to_R12,R1_transition_to_R12,R2_transition_to_R12,R12_transition_to_R12,misspecification_rate
0,1253756,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,1.000000e-11,0,0,0,0,1.000000e-11,0.001000,0,div5
1,1253699,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,1.000000e-03,0,0,0,0,1.000000e-03,0.000002,0,div5
2,1255036,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,1.000000e-03,0,0,0,0,1.000000e-03,0.001000,0,div5
3,1255770,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,4.642000e-05,0,0,0,0,4.642000e-05,0.001000,0,div5
4,1258270,5000000000,0,5,0,0.0184,0.0184,0.0184,0.0184,0.087737,...,0,1.000000e-03,0,0,0,0,1.000000e-03,0.001000,0,div5


In [ ]:
db_simTrajectories_df = pd.read_csv(miss_rate + "_" + str(sim_run_id) + '_simTrajectories.csv')
db_simTrajectories_df.head()

,Parameter_ID,Strategy_name,timepoint,Drug1_dosage,Spop,R1pop,R2pop,R12pop,misspecification_rate
0,1253356,CPM,0,1.0,5.000000e+09,0.0,5.0,0.00,div5
1,1253356,CPM,45,1.0,2.210000e+08,5070.0,183000.0,3.87,div5
2,1253356,CPM,90,1.0,9.780000e+06,11800.0,16100.0,18.80,div5
3,1253356,CPM,135,1.0,4.250000e+05,27100.0,1070.0,65.50,div5
4,1253356,CPM,180,1.0,1.850000e+04,62000.0,62.8,201.00,div5


add benefit boolean

In [ ]:
db_ECsurvival_df = pd.read_csv(miss_rate + "_" + str(sim_run_id) + '_ECsurvival.csv')
db_ECsurvival_df.head()

,Parameter_ID,Survival_CPM,Survival_DPM,EC_category,Bucket,DPM_days_improvement,DPM_percent_improvement,misspecification_rate
0,1253756,1845.0,1845.0,first_same_only,1.0_1.0_1.0_0.0,0.0,0.000000,div5
1,1253699,1442.0,1568.0,first_same_only,1.0_1.0_1.0_0.5,126.0,0.080357,div5
2,1255036,1845.0,1845.0,first_same_only,1.0_1.0_1.0_0.0,0.0,0.000000,div5
3,1255770,1306.0,1418.0,first_same_only,1.0_1.0_1.0_0.0,112.0,0.078984,div5
4,1258270,1143.0,1393.0,second_same_only,1.0_0.5_1.0_1.0,250.0,0.179469,div5


## development of script to process complete pipeline

In [ ]:
sim_output_dirs = !gsutil ls {sim_output_bucket}
misspecification_rates = []
for path in sim_output_dirs:
  final_part = path.rstrip('/').split('/')[-1]
  miss_rate = final_part.split('_')[0]
  misspecification_rates.append(miss_rate)


for miss_rate in misspecification_rates:
  print(miss_rate)
  sim_output_files = !gsutil ls gs://misspecification_dpm/{miss_rate}_atsim/
  valid_groups = validate_groups(sim_output_files)
  print(len(valid_groups)," Valid groups with all required fields:", valid_groups)
  for output_group in valid_groups:
    processed_files_exists = !gsutil ls {dbFile_output_dir}{miss_rate}{output_group}ECsurvival.csv

    if "ECsurvival" in processed_files_exists[0]:
      print(f'{output_group} already processed')
      continue
    print(f'processing {output_group}')

    sim_run_id = output_group.strip("_")
    group_paths = !gsutil ls gs://misspecification_dpm/{miss_rate}_atsim/*{output_group}*
    group_output_data = load_group_dataframes(group_paths, output_group)
    processed_output = process_group_output(group_output_data)
    process_sim_output(sim_run_id, miss_rate, processed_output['renamed_param'], processed_output['expanded_dosage'], processed_output['expanded_pop'], processed_output['renamed_stopt'], "./")
    !gsutil -m cp *.csv {dbFile_output_dir}
    !rm *.csv



In [ ]:
for path in group_paths:
  print(path)
  os.makedirs(failed_sim_output_dir + miss_rate + "/", exist_ok=True)
  !gsutil -m mv {path} {failed_sim_output_dir}/{miss_rate}/

# process all

In [ ]:
sim_output_dirs = !gsutil ls {sim_output_bucket}
misspecification_rates = []

for path in sim_output_dirs:
    final_part = path.rstrip('/').split('/')[-1]
    miss_rate = final_part.split('_')[0]
    misspecification_rates.append(miss_rate)

for miss_rate in misspecification_rates:
    print(miss_rate)
    sim_output_files = !gsutil ls gs://misspecification_dpm/{miss_rate}_atsim/
    valid_groups = validate_groups(sim_output_files)
    print(len(valid_groups), " Valid groups with all required fields:", valid_groups)

    for output_group in valid_groups:
        processed_files_exists = !gsutil ls {dbFile_output_dir}{miss_rate}{output_group}ECsurvival.csv

        if processed_files_exists and "ECsurvival" in processed_files_exists[0]:
            print(f'{output_group} already processed')
            continue

        print(f'processing {output_group}')
        sim_run_id = output_group.strip("_")
        group_paths = !gsutil ls gs://misspecification_dpm/{miss_rate}_atsim/*{output_group}*

        try:
            group_output_data = load_group_dataframes(group_paths, output_group)
        except Exception as e:
            print(f"Error loading dataframes for {output_group}: {e}")
            print(f"Moving sim results file to {failed_sim_output_dir}/{miss_rate}/")
            for path in group_paths:
              print(path)
              os.makedirs(failed_sim_output_dir + miss_rate + "/", exist_ok=True)
              !gsutil -m mv {path} {failed_sim_output_dir}/{miss_rate}/
            continue

        try:
            processed_output = process_group_output(group_output_data)
        except Exception as e:
            print(f"Error processing output for {output_group}: {e}")
            continue

        try:
            process_sim_output(
                sim_run_id,
                miss_rate,
                processed_output['renamed_param'],
                processed_output['expanded_dosage'],
                processed_output['expanded_pop'],
                processed_output['renamed_stopt'],
                "./"
            )
            !gsutil -m cp *.csv {dbFile_output_dir}
            !rm *.csv
        except Exception as e:
            print(f"Error saving output for {output_group}: {e}")
            continue

10x
290  Valid groups with all required fields: ['_053_', '_158_', '_225_', '_027_', '_301_', '_304_', '_146_', '_154_', '_138_', '_198_', '_230_', '_150_', '_279_', '_151_', '_108_', '_125_', '_260_', '_203_', '_300_', '_038_', '_083_', '_200_', '_095_', '_092_', '_050_', '_082_', '_117_', '_247_', '_216_', '_017_', '_006_', '_021_', '_101_', '_112_', '_290_', '_009_', '_070_', '_183_', '_118_', '_078_', '_181_', '_185_', '_289_', '_015_', '_132_', '_192_', '_007_', '_210_', '_088_', '_086_', '_010_', '_285_', '_152_', '_163_', '_257_', '_233_', '_128_', '_160_', '_293_', '_049_', '_308_', '_076_', '_167_', '_165_', '_204_', '_232_', '_213_', '_187_', '_139_', '_023_', '_145_', '_309_', '_265_', '_071_', '_262_', '_039_', '_055_', '_067_', '_276_', '_256_', '_012_', '_054_', '_248_', '_236_', '_280_', '_211_', '_020_', '_058_', '_286_', '_090_', '_120_', '_129_', '_275_', '_261_', '_243_', '_131_', '_013_', '_022_', '_179_', '_173_', '_048_', '_153_', '_075_', '_239_', '_172_', '_266_

# Process failed sim groups

In [ ]:
import os
from collections import defaultdict

root_dir = failed_sim_output_dir
summary = {}

for subdir in os.listdir(root_dir):
    subdir_path = os.path.join(root_dir, subdir)
    if os.path.isdir(subdir_path):
        id_counts = defaultdict(int)

        for file_name in os.listdir(subdir_path):
            if file_name.endswith(".csv"):
                parts = file_name.split("_")
                if len(parts) >= 6:
                    sim_run_id = parts[5]
                    id_counts[sim_run_id] += 1

        # Filter only IDs with exactly 4 files
        valid_ids = [sim_id for sim_id, count in id_counts.items() if count == 4]
        summary[subdir] = valid_ids

# Print summary
for miss_rate, ids in summary.items():
    print(f"Miss Rate: {miss_rate}")
    print(f"Unique Sim Run IDs with 4 files: {', '.join(ids) if ids else 'None'}\n")

Miss Rate: 5x
Unique Sim Run IDs with 4 files: 216, 212, 233, 224, 209, 202, 189, 180, 226, 177, 182, 194, 197, 207, 211, 218, 217, 213

Miss Rate: 10x
Unique Sim Run IDs with 4 files: 212, 199, 177, 214, 224, 195, 094, 218, 206, 235, 182, 217, 209, 194, 208, 197, 219, 189, 221

Miss Rate: 30x
Unique Sim Run IDs with 4 files: 205, 202, 209, 207, 215, 187, 247, 188, 198, 201, 199, 195, 220, 197, 218, 206, 196, 210

Miss Rate: div10
Unique Sim Run IDs with 4 files: 183, 181, 225, 203, 201, 191, 196, 215, 226, 207, 208, 194, 213, 192, 227, 218, 234

Miss Rate: div30
Unique Sim Run IDs with 4 files: 211, 198, 193, 181, 224, 208, 213, 200, 212, 197, 210, 216, 207, 188, 199, 222, 205

Miss Rate: div5
Unique Sim Run IDs with 4 files: 198, 021, 224, 267, 206, 248, 231, 271, 187, 277, 251, 280, 189, 289, 241, 216, 261, 177, 226, 205, 252



In [ ]:
import os

# Parameters
root_dir = failed_sim_output_dir
miss_rate = "div30"
sim_run_id = "211"



# Path to the chosen miss_rate directory
subdir_path = os.path.join(root_dir, miss_rate)

if not os.path.isdir(subdir_path):
    print(f"Directory for miss_rate '{miss_rate}' not found.")
else:
    # Find CSV files matching sim_run_id
    matching_files = [
        f for f in os.listdir(subdir_path)
        if f.endswith(".csv") and sim_run_id in f
    ]

    if not matching_files:
        print(f"No CSV files found for sim_run_id '{sim_run_id}' in '{miss_rate}'.")
    else:
        for file_name in matching_files:
            file_path = os.path.join(subdir_path, file_name)
            print(f"\n--- First 5 lines of {file_name} ---")
            with open(file_path, "r", encoding="utf-8") as file:
                for i, line in enumerate(file):
                    if i >= 5:
                        break
                    comma_count = line.count(",")
                    print(f"{line.strip()}  [commas: {comma_count}]")



--- First 5 lines of mis_PARset_default_PNAS_2drug_211_10000_para_result_dosage_20250107.csv ---
paramID,Strategy name,"(Drug1 dosage,Drug2 dosage) at t=0","(Drug1 dosage,Drug2 dosage) at t=45","(Drug1 dosage,Drug2 dosage) at t=90","(Drug1 dosage,Drug2 dosage) at t=135","(Drug1 dosage,Drug2 dosage) at t=180","(Drug1 dosage,Drug2 dosage) at t=225","(Drug1 dosage,Drug2 dosage) at t=270","(Drug1 dosage,Drug2 dosage) at t=315","(Drug1 dosage,Drug2 dosage) at t=360","(Drug1 dosage,Drug2 dosage) at t=405","(Drug1 dosage,Drug2 dosage) at t=450","(Drug1 dosage,Drug2 dosage) at t=495","(Drug1 dosage,Drug2 dosage) at t=540","(Drug1 dosage,Drug2 dosage) at t=585","(Drug1 dosage,Drug2 dosage) at t=630","(Drug1 dosage,Drug2 dosage) at t=675","(Drug1 dosage,Drug2 dosage) at t=720","(Drug1 dosage,Drug2 dosage) at t=765","(Drug1 dosage,Drug2 dosage) at t=810","(Drug1 dosage,Drug2 dosage) at t=855","(Drug1 dosage,Drug2 dosage) at t=900","(Drug1 dosage,Drug2 dosage) at t=945","(Drug1 dosage,Drug2 dosag